# Earnings Call Sentiment Backtest
**Does management's tone on an earnings call predict how the stock moves afterwards?**

This notebook scores each transcript with **FinBERT** (a free, open-source language model trained on financial text), pulls stock prices with **yfinance**, and tests whether more positive calls are followed by better stock performance *relative to the market*.

No API keys needed. Everything here is free.

### How to use it
1. Save each transcript as a `.txt` file named **`TICKER_YYYY-MM-DD.txt`**, using the Yahoo Finance ticker and the date of the call.
   - US stocks: `NVDA_2025-08-27.txt`, `AAPL_2025-07-31.txt`
   - Indian stocks need `.NS` (NSE): `INFY.NS_2025-07-17.txt`, `HDFCBANK.NS_2025-07-19.txt`
2. Go to **Runtime → Run all**. When the upload button appears in step 2, select all your transcript files.
3. Optional but faster: **Runtime → Change runtime type → T4 GPU**.

**Aim for 20+ transcripts** (e.g. 5 companies × 4 quarters). With fewer, any correlation you find is mostly noise.

## 1. Install and import

In [ ]:
!pip install -q transformers yfinance

import re, os, glob, warnings
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from scipy import stats
from transformers import pipeline
import torch

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)

BLUE, ORANGE, GREY = "#2F6DB5", "#E07B1F", "#8A93A3"
plt.rcParams.update({"figure.dpi": 130, "font.size": 11, "axes.titleweight": "bold",
                     "axes.spines.top": False, "axes.spines.right": False})
print("GPU available:", torch.cuda.is_available())

## 2. Upload transcripts
Select all your `TICKER_YYYY-MM-DD.txt` files at once.

In [ ]:
from google.colab import files
os.makedirs("transcripts", exist_ok=True)
uploaded = files.upload()
for name, data in uploaded.items():
    with open(os.path.join("transcripts", name), "wb") as f:
        f.write(data)

paths = sorted(glob.glob("transcripts/*.txt"))
events = []
for p in paths:
    m = re.match(r"(.+)_(\d{4}-\d{2}-\d{2})\.txt$", os.path.basename(p))
    if not m:
        print("Skipping (name doesn't match TICKER_YYYY-MM-DD.txt):", os.path.basename(p))
        continue
    events.append({"ticker": m.group(1).upper(), "date": pd.Timestamp(m.group(2)), "path": p})

print(f"Loaded {len(events)} transcripts")
pd.DataFrame(events)[["ticker", "date"]]

## 3. Score sentiment with FinBERT
Each transcript is split into sentences. FinBERT labels every sentence positive, negative or neutral.
The **net sentiment** score = average positive probability − average negative probability (ranges roughly −1 to +1).

The notebook also scores the **prepared remarks** and the **analyst Q&A** separately, since management's scripted tone and their unscripted answers often differ.

In [ ]:
device = 0 if torch.cuda.is_available() else -1
finbert = pipeline("text-classification", model="ProsusAI/finbert", top_k=None,
                   truncation=True, max_length=512, device=device)

QA_MARKERS = re.compile(r"question[- ]and[- ]answer|open (up )?the (line|floor|call) for questions|"
                        r"first question|we will now (begin|take) questions|q&a session", re.I)

def split_sentences(text):
    text = re.sub(r"\s+", " ", text)
    sents = re.split(r"(?<=[.!?])\s+(?=[A-Z\"'])", text)
    out = []
    for s in sents:
        s = re.sub(r"^[A-Z][\w .,'&-]{0,60}:\s*", "", s).strip()   # drop "Speaker Name, Title:" prefixes
        if len(s.split()) >= 6 and not s.lower().startswith(("operator", "thank you", "thanks")):
            out.append(s)
    return out

def score_sentences(sents):
    if not sents:
        return pd.DataFrame(columns=["sentence", "positive", "negative", "neutral"])
    results = finbert(sents, batch_size=32)
    rows = []
    for s, r in zip(sents, results):
        d = {x["label"]: x["score"] for x in r}
        rows.append({"sentence": s, "positive": d.get("positive", 0),
                     "negative": d.get("negative", 0), "neutral": d.get("neutral", 0)})
    return pd.DataFrame(rows)

def net(df):
    return float(df["positive"].mean() - df["negative"].mean()) if len(df) else np.nan

sentence_scores = {}
for e in events:
    text = open(e["path"], encoding="utf-8", errors="ignore").read()
    m = QA_MARKERS.search(text)
    remarks_txt, qa_txt = (text[:m.start()], text[m.start():]) if m else (text, "")
    rem = score_sentences(split_sentences(remarks_txt)); rem["part"] = "Prepared remarks"
    qa = score_sentences(split_sentences(qa_txt));       qa["part"] = "Q&A"
    allsc = pd.concat([rem, qa], ignore_index=True)
    sentence_scores[(e["ticker"], e["date"])] = allsc
    e.update({
        "n_sentences": len(allsc),
        "net_sentiment": net(allsc),
        "remarks_sentiment": net(rem),
        "qa_sentiment": net(qa),
        "pct_negative": float((allsc[["positive","negative","neutral"]].idxmax(axis=1) == "negative").mean()),
    })
    print(f"{e['ticker']:>12} {e['date'].date()}  net={e['net_sentiment']:+.3f}  "
          f"remarks={e['remarks_sentiment']:+.3f}  Q&A={e['qa_sentiment']:+.3f}  ({len(allsc)} sentences)")

## 4. Measure the stock's reaction
For each call, the notebook measures the stock's return from the **last close before the call** to **3 and 10 trading days later**.
Starting from the prior close captures the reaction whether the call happened before the market opened or after it closed.

To strip out general market moves, it also calculates the **abnormal return**: the stock's return minus its benchmark's
(S&P 500 for US stocks, Nifty 50 for `.NS`/`.BO` stocks). This is the number that matters.

In [ ]:
HORIZONS = [3, 10]

def benchmark_for(ticker):
    return "^NSEI" if ticker.endswith((".NS", ".BO")) else "^GSPC"

_price_cache = {}
def closes(ticker, start, end):
    key = (ticker, start, end)
    if key not in _price_cache:
        h = yf.Ticker(ticker).history(start=start, end=end, auto_adjust=True)["Close"]
        h.index = h.index.tz_localize(None).normalize()
        _price_cache[key] = h
    return _price_cache[key]

def window_returns(series, call_date):
    before = series[series.index < call_date]
    after = series[series.index >= call_date]
    if before.empty or len(after) < max(HORIZONS):
        return {h: np.nan for h in HORIZONS}
    base = before.iloc[-1]
    return {h: after.iloc[h - 1] / base - 1 for h in HORIZONS}

for e in events:
    start = (e["date"] - pd.Timedelta(days=15)).strftime("%Y-%m-%d")
    end = (e["date"] + pd.Timedelta(days=30)).strftime("%Y-%m-%d")
    try:
        stock = window_returns(closes(e["ticker"], start, end), e["date"])
        bench = window_returns(closes(benchmark_for(e["ticker"]), start, end), e["date"])
    except Exception as ex:
        print("Price download failed for", e["ticker"], ex)
        stock = bench = {h: np.nan for h in HORIZONS}
    for h in HORIZONS:
        e[f"ret_{h}d"] = stock[h]
        e[f"abn_{h}d"] = stock[h] - bench[h]

results = pd.DataFrame(events).drop(columns="path")
missing = results[results["abn_10d"].isna()]
if len(missing):
    print("No price data (check the ticker, or the call may be too recent):", list(missing["ticker"]))

show = results.copy()
for c in [c for c in show.columns if c.startswith(("ret_", "abn_", "pct_"))]:
    show[c] = (show[c] * 100).round(2)
show[["ticker", "date", "net_sentiment", "remarks_sentiment", "qa_sentiment", "ret_3d", "abn_3d", "ret_10d", "abn_10d"]].round(3)

## 5. Does sentiment predict returns?
Two correlation measures:
- **Pearson**: linear relationship, sensitive to outliers.
- **Spearman**: do higher-sentiment calls *rank* higher on returns? More robust with small samples.

A **p-value below 0.05** is the usual bar for "probably not just chance". With a small sample, expect it to be higher. That is an honest and perfectly good finding to report.

In [ ]:
data = results.dropna(subset=["abn_10d", "net_sentiment"])
n = len(data)
print(f"Events with complete data: {n}")
if n < 20:
    print("⚠️  Fewer than 20 events: treat these results as exploratory, not conclusive.\n")

rows = []
for sent_col in ["net_sentiment", "remarks_sentiment", "qa_sentiment"]:
    for h in HORIZONS:
        d = data.dropna(subset=[sent_col, f"abn_{h}d"])
        if len(d) < 3:
            continue
        pr, pp = stats.pearsonr(d[sent_col], d[f"abn_{h}d"])
        sr, sp = stats.spearmanr(d[sent_col], d[f"abn_{h}d"])
        rows.append({"sentiment measure": sent_col, "horizon": f"{h} days",
                     "pearson r": round(pr, 3), "pearson p": round(pp, 3),
                     "spearman ρ": round(sr, 3), "spearman p": round(sp, 3), "n": len(d)})
corr = pd.DataFrame(rows)
corr

## 6. Charts for your LinkedIn post

In [ ]:
d = data.copy()
d["label"] = d["ticker"].str.replace(".NS", "", regex=False).str.replace(".BO", "", regex=False) + " " + d["date"].dt.strftime("%b %y")
x, y = d["net_sentiment"], d["abn_10d"] * 100

fig, ax = plt.subplots(figsize=(10, 6))
colors = np.where(y >= 0, BLUE, ORANGE)
ax.scatter(x, y, c=colors, s=70, edgecolor="white", linewidth=1, zorder=3)
for xi, yi, lab in zip(x, y, d["label"]):
    ax.annotate(lab, (xi, yi), xytext=(5, 4), textcoords="offset points", fontsize=8.5, color="#444")
if len(d) >= 3:
    slope, intercept, r, p, _ = stats.linregress(x, y)
    xs = np.linspace(x.min(), x.max(), 50)
    ax.plot(xs, intercept + slope * xs, color=GREY, linestyle="--", linewidth=1.5,
            label=f"Trend line (r = {r:.2f}, p = {p:.2f}, n = {len(d)})")
    ax.legend(loc="upper left", frameon=False)
ax.axhline(0, color="#ccc", linewidth=1, zorder=1)
ax.set_xlabel("Net call sentiment (FinBERT)  →  more positive")
ax.set_ylabel("10-day return vs. market (%)")
ax.set_title("Does a more positive earnings call lead to better stock performance?")
plt.tight_layout(); plt.savefig("sentiment_vs_returns.png", dpi=200, bbox_inches="tight"); plt.show()

# Prepared remarks vs Q&A tone
dd = d.dropna(subset=["qa_sentiment"]).sort_values("net_sentiment")
if len(dd):
    fig, ax = plt.subplots(figsize=(10, max(4, 0.45 * len(dd))))
    yy = np.arange(len(dd))
    ax.barh(yy - 0.2, dd["remarks_sentiment"], height=0.4, color=BLUE, label="Prepared remarks")
    ax.barh(yy + 0.2, dd["qa_sentiment"], height=0.4, color=ORANGE, hatch="//", edgecolor="white", label="Analyst Q&A")
    ax.set_yticks(yy); ax.set_yticklabels(dd["label"])
    ax.axvline(0, color="#999", linewidth=1)
    ax.set_xlabel("Net sentiment")
    ax.set_title("Management sounds more upbeat in scripted remarks than in live Q&A")
    ax.legend(frameon=False, loc="lower right")
    plt.tight_layout(); plt.savefig("remarks_vs_qa.png", dpi=200, bbox_inches="tight"); plt.show()
    gap = (dd["remarks_sentiment"] - dd["qa_sentiment"]).mean()
    print(f"Average tone drop from prepared remarks to Q&A: {gap:+.3f}"
          + ("  (if this is negative, retitle the chart!)" if gap < 0 else ""))

## 7. Most positive and negative sentences per call
Useful for sanity-checking the model and for talking points in interviews.

In [ ]:
for (ticker, date), sc in sentence_scores.items():
    if sc.empty:
        continue
    sc = sc.assign(net=sc["positive"] - sc["negative"])
    print(f"\n=== {ticker} {date.date()} ===")
    print("Most positive:")
    for s in sc.nlargest(2, "net")["sentence"]:
        print("  +", s[:220])
    print("Most negative:")
    for s in sc.nsmallest(2, "net")["sentence"]:
        print("  −", s[:220])

## 8. Save and download results

In [ ]:
results.to_csv("earnings_sentiment_results.csv", index=False)
corr.to_csv("correlations.csv", index=False)
for f in ["earnings_sentiment_results.csv", "correlations.csv", "sentiment_vs_returns.png", "remarks_vs_qa.png"]:
    if os.path.exists(f):
        files.download(f)

## Interpreting your results honestly
- **Correlation isn't causation**, and a handful of events can produce a strong-looking r by chance. Check the p-value and n.
- **Markets react to numbers vs. expectations** (did earnings beat analyst estimates?), which this model doesn't see. Tone is just one signal.
- **FinBERT reads sentences in isolation.** It can miss sarcasm, context, and "less bad than feared" news.
- **Ideas to extend it:** add more quarters, control for earnings surprises, compare FinBERT's scores with the AI analyzer's scores on the same calls, or test whether the *change* in tone from last quarter matters more than the level.

Whatever the result, a clearly explained "no strong relationship in my sample" is a credible finding. Explaining *why* is what impresses people.